# 01 - The Data Factory

In [1]:
# Setup & Configuration
import sys
import os
import glob
import json
import random
import re
import warnings
from pathlib import Path
from dotenv import load_dotenv
from tqdm.auto import tqdm
import pandas as pd

notebook_dir = Path.cwd()
if notebook_dir.name == "notebooks":
    project_root = notebook_dir.parent
else:
    project_root = notebook_dir

sys.path.insert(0, str(project_root))
os.chdir(project_root)

from src.services.llm_services import load_config, get_llm, validate_api_keys, print_config_summary

load_dotenv()
config = load_config("src/config/config.yaml")
validate_api_keys(config)
print_config_summary(config)

llm = get_llm(config)
print(f"LLM initialized: {config['llm_provider']} / {config['llm_model']}")

d:\Zuu Crew\AEE\0351 Romen Ranasingha\zuucrew-mini-project-1\src\services\llm_services.py:377: UserWarning: ⚠️  OPENAI_API_KEY not found in environment
  warnings.warn(f"⚠️  {key} not found in environment")
d:\Zuu Crew\AEE\0351 Romen Ranasingha\zuucrew-mini-project-1\src\services\llm_services.py:377: UserWarning: ⚠️  OPENROUTER_API_KEY not found in environment
  warnings.warn(f"⚠️  {key} not found in environment")
d:\Zuu Crew\AEE\0351 Romen Ranasingha\zuucrew-mini-project-1\src\services\llm_services.py:377: UserWarning: ⚠️  GROQ_API_KEY not found in environment
  warnings.warn(f"⚠️  {key} not found in environment")
d:\Zuu Crew\AEE\0351 Romen Ranasingha\zuucrew-mini-project-1\src\services\llm_services.py:377: UserWarning: ⚠️  COHERE_API_KEY not found in environment
  warnings.warn(f"⚠️  {key} not found in environment")


✅ Config loaded:
  LLM: gemini / gemini-2.5-flash
  Embeddings: sbert / sentence-transformers/all-MiniLM-L6-v2
  Temperature: 0.2
  Artifacts: ./artifacts
LLM initialized: gemini / gemini-2.5-flash


## Ingestion & Cleaning

In [2]:
from pypdf import PdfReader

pdf_path = Path(config["data_root"]) / "pdfs" / "2024AnnualReport.pdf"

def load_pdf(path):
    reader = PdfReader(path)
    text = ""
    for page in reader.pages:
        content = page.extract_text()
        if content:
            text += content + "\n"
    return text

def clean_text(text):
    text = re.sub(r'\s+', ' ', text).strip()
    return text

if not pdf_path.exists():
    print(f"PDF not found at {pdf_path}.")
else:
    raw_text = load_pdf(pdf_path)
    cleaned_text = clean_text(raw_text)
    print(f"Total characters: {len(cleaned_text)}")
    print(f"Sample: {cleaned_text[:200]}...")

Total characters: 624497
Sample: On Our Way 2024 ANNUAL REPORT Uber’s Mission We reimagine the way the world moves for the better We are Uber. The go-getters. The kind of people who are relentless about our mission to help people go ...


## Chunking Strategy (1500 chars)

In [3]:
CHUNK_SIZE = 1500
OVERLAP = 0

def chunk_text(text, chunk_size=CHUNK_SIZE):
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start : start + chunk_size])
        start += chunk_size
    return chunks

chunks = chunk_text(cleaned_text)
print(f"Created {len(chunks)} chunks.")
print(f"Sample chunk 0: {chunks[0][:100]}...")

Created 417 chunks.
Sample chunk 0: On Our Way 2024 ANNUAL REPORT Uber’s Mission We reimagine the way the world moves for the better We ...


## The Generation Loop

In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()

q_gen_template = """
You are a professional financial analyst auditing Uber's 2024 performance.
Based ONLY on the text chunk provided, generate 10 diverse Question-Answer pairs.

Categories to cover:
1. Hard Facts (numbers, specific dates, legal entities)
2. Strategic Summaries (management goals, competitive risks, long-term outlook)
3. Stylistic/Creative (professional tone, describing complex terms formally)

CRITICAL: Return ONLY a JSON list of objects. Each object must have EXACTLY these two keys: "question" and "answer".
Do not include any other text or reasoning outside the JSON block.

TEXT CHUNK:
{context}
"""

q_gen_prompt = ChatPromptTemplate.from_template(template=q_gen_template)

dataset = []
print(f"Generating Q/A pairs for {len(chunks)} chunks...")

# Process chunks
process_chunks = chunks 

for i, chunk in tqdm(enumerate(process_chunks), total=len(process_chunks)):
    if not chunk.strip():
        continue
    try:
        formatted_prompt = q_gen_prompt.format(context=chunk)
        response = llm.invoke(formatted_prompt)
        raw_content = response.content
        
        raw_content = re.sub(r'^```json\n?', '', raw_content)
        raw_content = re.sub(r'\n?```$', '', raw_content).strip()

        try:
            result = json.loads(raw_content)
        except json.JSONDecodeError:
            try:
                result = parser.parse(raw_content)
            except:
                # print(f"Failed to parse chunk {i}")
                continue
        
        qa_list = []
        if isinstance(result, list):
            qa_list = result
        elif isinstance(result, dict):
            for key in ["qa_pairs", "questions", "data", "results", "pairs"]:
                if key in result and isinstance(result[key], list):
                    qa_list = result[key]
                    break

        for item in qa_list:
            q = item.get("question") or item.get("Question")
            a = item.get("answer") or item.get("Answer")
            
            if q and a and len(str(q)) > 10 and len(str(a)) > 10:
                dataset.append({
                    "instruction": str(q).strip(),
                    "input": "",
                    "output": str(a).strip(),
                    "source_chunk": i
                })

    except Exception as e:
        # print(f"LLM error on chunk {i}: {e}")
        pass

print(f"Total valid Q/A pairs: {len(dataset)}")

Generating Q/A pairs for 417 chunks...


  0%|          | 0/417 [00:00<?, ?it/s]

Total valid Q/A pairs: 2237


## Train/Test Split & Storage

In [5]:
if not dataset:
    print("No data generated.")
else:
    random.seed(42)
    random.shuffle(dataset)

    split_idx = int(len(dataset) * 0.8)
    train_data = dataset[:split_idx]
    test_data = dataset[split_idx:]

    print(f"Train set: {len(train_data)}")
    print(f"Test set: {len(test_data)}")

    def save_jsonl(data, filename):
        with open(filename, 'w', encoding='utf-8') as f:
            for entry in data:
                json.dump(entry, f)
                f.write('\n')

    output_dir = Path(config["artifacts_root"]) / "data_factory"
    output_dir.mkdir(parents=True, exist_ok=True)

    save_jsonl(train_data, output_dir / "train.jsonl")
    save_jsonl(test_data, output_dir / "golden_test_set.jsonl")

    print(f"Saved artifacts to {output_dir}")

Train set: 1789
Test set: 448
Saved artifacts to artifacts\data_factory
